In [8]:
import inspect
print(inspect.signature(IGTD.__init__))


(self, problem: Optional[str] = None, transformer=MinMaxScaler(), verbose: Optional[bool] = None, scale: Optional[List[int]] = [6, 6], fea_dist_method: Optional[str] = 'Pearson', image_dist_method: Optional[str] = 'Euclidean', error: Optional[str] = 'squared', max_step: Optional[int] = 1000, val_step: Optional[int] = 50, switch_t: Optional[int] = 0, min_gain: Optional[float] = 1e-05, zoom: Optional[int] = 1, format='png', cmap='gray', random_seed: Optional[int] = 1)


In [9]:
# feature_dim 需要通过查看模型源码或动态打印模型来获得
import torch
from torchvision.models import (
    efficientnet_v2_m, EfficientNet_V2_M_Weights, # V1
    resnext50_32x4d, ResNeXt50_32X4D_Weights, # V2
    mobilenet_v3_large, MobileNet_V3_Large_Weights, # V2
    densenet161, DenseNet161_Weights # V1
)

weights = EfficientNet_V2_M_Weights.IMAGENET1K_V1
model = efficientnet_v2_m(weights=weights)

# weights = ResNeXt50_32X4D_Weights.IMAGENET1K_V2
# model = resnext50_32x4d(weights=weights)

# weights = MobileNet_V3_Large_Weights.IMAGENET1K_V2
# model = mobilenet_v3_large(weights=weights)

# weights = DenseNet161_Weights.IMAGENET1K_V1
# model = densenet161(weights=weights)

print(model)

print(model.classifier)
# print(model.fc)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 24, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): FusedMBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): FusedMBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  

In [ ]:
# 1. 导入库
import pandas as pd
import numpy as np
from TINTOlib.igtd import IGTD
import matplotlib.pyplot as plt

# 2. 读取 iris 数据
data_path = '../data/iris/iris.data'

# iris.data 文件没有列名，header=None
df = pd.read_csv(data_path, header=None)

# 加上列名（iris 数据集的官方列名）
df.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'class']

# 把类别转换为数值（TINTOlib 需要数值型标签）
df['class_num'] = df['class'].map({
    'Iris-setosa': 0,
    'Iris-versicolor': 1,
    'Iris-virginica': 2
})

# 提取特征和标签
X = df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']].values
y = df['class_num'].values

print(f"数据形状: {X.shape}")
print(f"样本数: {X.shape[0]}, 特征数: {X.shape[1]}")
print(f"类别分布: {np.bincount(y)}")

# 3. 使用 IGTD 生成图像
print("\n正在使用 IGTD 转换数据...")
igtd = IGTD(
    problem='supervised',      # 分类任务
    max_step=100,                # 优化迭代次数
    verbose=True,
    scale=[32, 32],
    random_seed=1
)

# 生成图像
images = igtd.fit_transform(X, y)

print(f"生成的图像形状: {images.shape}")  # 应该是 (n_samples, 32, 32, 3)

# 4. 可视化生成的图像
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()

for i in range(10):
    axes[i].imshow(images[i])
    axes[i].set_title(f"Class: {df['class'].iloc[i]}")
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('../output/iris_igtd_samples.png', dpi=150)
plt.show()

print("\n✅ 完成！图像已生成并保存到 output 文件夹。")

d:\Downloads\anaconda3\envs\CHM9360\Lib\site-packages\TINTOlib\abstractImageMethod.py:43: FutureWarning: Problem type supervised will be deprecated. Instead use classification.
  warnings.warn("Problem type supervised will be deprecated. Instead use classification.",FutureWarning)


数据形状: (150, 4)
样本数: 150, 特征数: 4
类别分布: [50 50 50]

正在使用 IGTD 转换数据...


Preparing Data:   0%|          | 0/100 [00:00<?, ?it/s]


TypeError: data must be a string (file path) or a pandas DataFrame.

In [19]:
def generate_igtd_images(df, feature_cols, target_col, output_folder, img_size=32):
    """
    使用 IGTD 生成图像并返回 numpy 数组
    
    参数:
        df: DataFrame
        feature_cols: 特征列名列表
        target_col: 标签列名
        output_folder: 输出文件夹
        img_size: 图像尺寸
    """
    import os
    from PIL import Image
    
    # 准备数据
    df_igtd = df[feature_cols + [target_col]]
    
    # 创建输出文件夹
    os.makedirs(output_folder, exist_ok=True)
    
    # 生成图像
    igtd = IGTD(
        problem='supervised',
        max_step=100,
        verbose=True,
        scale=[img_size, img_size],
        random_seed=1
    )
    
    igtd.fit_transform(df_igtd, output_folder)
    
    # 加载生成的图像
    image_files = sorted([f for f in os.listdir(output_folder) if f.endswith('.png')])
    images = []
    for img_file in image_files:
        img = Image.open(os.path.join(output_folder, img_file))
        images.append(np.array(img))
    
    return np.array(images)

# 使用示例
images = generate_igtd_images(
    df=df,
    feature_cols=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'],
    target_col='class_num',
    output_folder='../output/igtd_images',
    img_size=32
)

print(f"生成的图像形状: {images.shape}")

d:\Downloads\anaconda3\envs\CHM9360\Lib\site-packages\TINTOlib\abstractImageMethod.py:43: FutureWarning: Problem type supervised will be deprecated. Instead use classification.
  warnings.warn("Problem type supervised will be deprecated. Instead use classification.",FutureWarning)
Generating and saving the synthetic images:  44%|████▎     | 43.533333333333374/100 [00:00<00:00, 428.33it/s]

Data successfully loaded.


Images generated and saved: 100%|██████████| 100.0/100 [00:00<00:00, 231.44it/s]                             

Fit-Transform process completed.
生成的图像形状: (0,)
